# Analyzing results and saving data

This file can be used to see the results and plots from a specific run, and to save the results in csv files for final plotting through "plots.ipynb".

Edit the settings to decide which scenario to run.

In [ ]:
import logging
import warnings
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# SETTINGS
save_to_csv = True
path = '../test_runs/'

In [ ]:
SCENARIOS = [
    # (display_name, case_path, file_code, discountrate, cost_loop, gas_price_4, gas_price_1)
    ('BASE',         'base',         'B_',  0.04, False, False, False),
    ('SC',       'sudden',       'S_',  0.04, False, False, False),
    ('GC',      'gradual',      'G_',  0.04, False, False, False),
    ('BASE NZE',     'base_NZE',     'BNZE_', 0.04, False, False, False),
    ('SC NZE',   'sudden_NZE',   'SNZE_', 0.04, False, False, False),
    ('GC NZE',  'gradual_NZE',  'GNZE_', 0.04, False, False, False),
    # discount rate sensitivity:
    ('BASE 7%',     'base_007',     'B_', 0.07, False, False, False),
    ('SC 7%',   'sudden_007',   'S_',  0.07, False, False, False),
    ('GC 7%',  'gradual_007',  'G_',  0.07, False, False, False),
    ('BASE 10%',     'base_010',     'B_', 0.10, False, False, False),
    ('SC 10%',   'sudden_010',   'S_',  0.10, False, False, False),
    ('GC 10%',  'gradual_010',  'G_',  0.10, False, False, False),
    # gas price sensitivity:
    ('BASE GAS LOW',   'base_gas_low',   'B_', 0.04, False, False,  True),
    ('BASE GAS HIGH',   'base_gas_high',   'B_', 0.04, False, True,  False),
    ('SC GAS LOW',   'sudden_gas_low',   'S_',  0.04, False, False,  True),
    ('SC GAS HIGH',   'sudden_gas_high',   'S_',  0.04, False, True,  False),
    # technology cost sensitivity:
    ('BASE NEW COST', 'base_cost', 'B_', 0.04, True, False, False),
    ('SC NEW COST', 'sudden_cost', 'S_', 0.04, True, False, False),
    ('GC NEW COST', 'gradual_cost', 'G_', 0.04, True, False, False),
    # weather sensitivity:
    ('BASE 2011', 'base_2011', 'B_', 0.04, False, False, False),
    ('SC 2011', 'sudden_2011', 'S_', 0.04, False, False, False),
    ('BASE 2018', 'base_2018', 'B_', 0.04, False, False, False),
    ('SC 2018', 'sudden_2018', 'S_', 0.04, False, False, False),
    # 4 nodes:
    ('GC NZE 4', 'gradual_NZE_4', 'GNZE_', 0.04, False, False, False),

]

In [ ]:
# RETREIVING COSTS FOR COST SENSITIVITY ANALYSIS
# Dictionary for cost_year, 2024 has cost year 2020, 2025 - 2029 has cost year 2025, 2030 - 2034 has cost year 2030, and 2035 - 2040 has cost year 2035
cost_years = {
    2024: '2020',
    2025: '2025',
    2026: '2025',
    2027: '2025',
    2028: '2025',
    2029: '2025',
    2030: '2030',
    2031: '2030',
    2032: '2030',
    2033: '2030',
    2034: '2030',
    2035: '2035',
    2036: '2035',
    2037: '2035',
    2038: '2035',
    2039: '2035',
    2040: '2035',
}   


scale_cost_sudden = {
    ('OCGT', 2025): 1.0,
    ('OCGT', 2026): 1.0,
    ('OCGT', 2027): 0.20,
    ('OCGT', 2028): 0.20,
    ('OCGT', 2029): 0.20,
    ('OCGT', 2030): 0.20,
    ('OCGT', 2031): 0.20,
    ('OCGT', 2032): 0.20,
    ('OCGT', 2033): 0.20,
    ('OCGT', 2034): 0.20,
    ('OCGT', 2035): 0.20,
    ('OCGT', 2036): 0.20,
    ('OCGT', 2037): 0.20,
    ('OCGT', 2038): 0.20,
    ('OCGT', 2039): 0.20,
    ('OCGT', 2040): 0.20,

    ('CCGT', 2025): 1.0,
    ('CCGT', 2026): 1.0,
    ('CCGT', 2027): 0.20,
    ('CCGT', 2028): 0.20,
    ('CCGT', 2029): 0.20,
    ('CCGT', 2030): 0.20,
    ('CCGT', 2031): 0.20,
    ('CCGT', 2032): 0.20,
    ('CCGT', 2033): 0.20,
    ('CCGT', 2034): 0.20,
    ('CCGT', 2035): 0.20,
    ('CCGT', 2036): 0.20,
    ('CCGT', 2037): 0.20,
    ('CCGT', 2038): 0.20,
    ('CCGT', 2039): 0.20,
    ('CCGT', 2040): 0.20,

    ('oil', 2025): 1.0,
    ('oil', 2026): 1.0,
    ('oil', 2027): 0.29,
    ('oil', 2028): 0.29,
    ('oil', 2029): 0.29,
    ('oil', 2030): 0.29,
    ('oil', 2031): 0.29,
    ('oil', 2032): 0.29,
    ('oil', 2033): 0.29,
    ('oil', 2034): 0.29,
    ('oil', 2035): 0.29,
    ('oil', 2036): 0.29,
    ('oil', 2037): 0.29,
    ('oil', 2038): 0.29,
    ('oil', 2039): 0.29,
    ('oil', 2040): 0.29,
}

scale_cost_gradual = {
    ('OCGT', 2025): 0.79,
    ('OCGT', 2026): 0.65,
    ('OCGT', 2027): 0.55,
    ('OCGT', 2028): 0.48,
    ('OCGT', 2029): 0.43,
    ('OCGT', 2030): 0.38,
    ('OCGT', 2031): 0.35,
    ('OCGT', 2032): 0.32,
    ('OCGT', 2033): 0.29,
    ('OCGT', 2034): 0.27,
    ('OCGT', 2035): 0.25,
    ('OCGT', 2036): 0.24,
    ('OCGT', 2037): 0.22,
    ('OCGT', 2038): 0.21,
    ('OCGT', 2039): 0.20,
    ('OCGT', 2040): 0.20,

    ('CCGT', 2025): 0.79,
    ('CCGT', 2026): 0.65,
    ('CCGT', 2027): 0.55,
    ('CCGT', 2028): 0.48,
    ('CCGT', 2029): 0.43,
    ('CCGT', 2030): 0.38,
    ('CCGT', 2031): 0.35,
    ('CCGT', 2032): 0.32,
    ('CCGT', 2033): 0.29,
    ('CCGT', 2034): 0.27,
    ('CCGT', 2035): 0.25,
    ('CCGT', 2036): 0.24,
    ('CCGT', 2037): 0.22,
    ('CCGT', 2038): 0.21,
    ('CCGT', 2039): 0.20,
    ('CCGT', 2040): 0.20,

    ('oil', 2025): 0.86,
    ('oil', 2026): 0.75,
    ('oil', 2027): 0.67,
    ('oil', 2028): 0.60,
    ('oil', 2029): 0.55,
    ('oil', 2030): 0.50,
    ('oil', 2031): 0.46,
    ('oil', 2032): 0.43,
    ('oil', 2033): 0.40,
    ('oil', 2034): 0.37,
    ('oil', 2035): 0.35,
    ('oil', 2036): 0.33,
    ('oil', 2037): 0.32,
    ('oil', 2038): 0.30,
    ('oil', 2039): 0.29,
    ('oil', 2040): 0.29,
}


In [ ]:
# Colors
red1 = '#891D2D'
red2 = '#BA3B31'
orange = '#F58221'
yellow = '#FCAF19'
brown = '#440A15'
brown2 = '#B45419'
purple1 = '#3B1053'
purple2 = '#76518E'
purple3 = '#B69DC7'
teal1 = '#032838'
teal2 = '#154655'
teal3 = '#527D77'
teal4 = '#8DB5AF'
teal1 = '#294839'
green1 = '#6DA08C'
green2 = '#6E966E'
green3 = '#A3BDA3'
beige1 = '#7A693B'
beige2 = '#A89677'
beige3 = '#D2CDAD'
grey1 = '#E7E7E7'
grey2 = '#D7D7D7'
grey3 = '#C6C6C6'
grey4 = '#939393'
blue1 = '#3EA1C0'

In [ ]:
# Helper functions

def scale_costs(year, case):
    """
    Scales the costs for OCGT, CCGT and oil based on the case (sudden or gradual).  
    Parameters:
    cost_year (str): The year for which the costs are being scaled.
    case (str): The case for which the costs are being scaled ('Sudden' or 'Gradual', or 'sudden' or 'gradual).
    Returns:
    dict: A dictionary with the scaled costs for OCGT, CCGT and oil.
    """
    scale_costs = {}
    if case.lower() == 'sudden':
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = scale_cost_sudden[(tech, year)]
    elif case.lower() == 'gradual':
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = scale_cost_gradual[(tech, year)]
    else: # Nothing to scale if case is not sudden or gradual
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = 1.0
    return scale_costs  

def get_power_prod(year):
    network = networks[year]
    carrier_list = network.generators.carrier.unique()
    carrier_list_2 = network.storage_units.carrier.unique()
    carriers = np.concatenate((carrier_list, carrier_list_2), axis=None)
    production_data = {}
    first_date = "2013-01-01"
    second_date = "2013-12-31"
 
    production_data["CCGT"] = get_snapshot_generation(year, first_date, second_date, "CCGT").sum() /1e6
    production_data["OCGT"] = get_snapshot_generation(year, first_date, second_date, "OCGT").sum() /1e6
    production_data["oil"] = get_snapshot_generation(year, first_date, second_date, "oil").sum() /1e6
    production_data["geothermal"] = get_snapshot_generation(year, first_date, second_date, "geothermal").sum() /1e6
    production_data["hydro"] = get_snapshot_generation(year, first_date, second_date, "hydro").sum() /1e6
    production_data["onwind"] = get_snapshot_generation(year, first_date, second_date, "onwind").sum() /1e6
    production_data["solar"] = get_snapshot_generation(year, first_date, second_date, "solar").sum() /1e6
    production_data["biomass"] = get_snapshot_generation(year, first_date, second_date, "biomass").sum() /1e6
    production_data["ror"] = get_snapshot_generation(year, first_date, second_date, "ror").sum() /1e6
 
    df = pd.DataFrame([production_data]) # in TWh
    df['hydro'] += df.pop('ror')
    return df

def get_power_prod_hydro(year):
    #print(year)
    network = networks[year]
    carrier_list = network.generators.carrier.unique()
    carrier_list_2 = network.storage_units.carrier.unique()
    carriers = np.concatenate((carrier_list, carrier_list_2), axis=None)
    production_data = {}
    first_date = "2013-01-01"
    second_date = "2013-12-31"
    # for carrier in carriers:
    #     production_data[carrier] = get_snapshot_generation(year, first_date, second_date, carrier).sum() /1e6
 
    production_data["CCGT"] = get_snapshot_generation(year, first_date, second_date, "CCGT").sum() /1e6
    production_data["OCGT"] = get_snapshot_generation(year, first_date, second_date, "OCGT").sum() /1e6
    production_data["oil"] = get_snapshot_generation(year, first_date, second_date, "oil").sum() /1e6
    production_data["geothermal"] = get_snapshot_generation(year, first_date, second_date, "geothermal").sum() /1e6
    production_data["hydro"] = get_snapshot_generation(year, first_date, second_date, "hydro").sum() /1e6
    production_data["onwind"] = get_snapshot_generation(year, first_date, second_date, "onwind").sum() /1e6
    production_data["solar"] = get_snapshot_generation(year, first_date, second_date, "solar").sum() /1e6
    production_data["biomass"] = get_snapshot_generation(year, first_date, second_date, "biomass").sum() /1e6
    production_data["ror"] = get_snapshot_generation(year, first_date, second_date, "ror").sum() /1e6
    #production_data["load"] = get_snapshot_generation(year, first_date, second_date, "load").sum() /1e6
 
    df = pd.DataFrame([production_data]) # in TWh
    return df

def get_demand(year): 
    network = networks[year]
    el_demand =network.loads_t.p_set
    regional_demand = pd.DataFrame(el_demand.sum()/1e6)
    return regional_demand.sum()

def total_production(year):
    prod = pd.DataFrame(get_power_prod(year))
    #prod.drop('load', axis=1, inplace=True)
    return prod.sum().sum()

def get_power_mix(year):
    total_prod = total_production(year)
    prod = get_power_prod(year)
    prod_series = prod.iloc[0]#.drop('load', errors='ignore')
    fractions = prod_series / total_prod

    df = pd.DataFrame(fractions).transpose()
    df.index = [year]  
    return df

def custom_autopct(pct):
    return ('%1.1f%%' % pct) if pct > 0 else ''

def get_emissions(year):
    network = networks[year]
    emissions = network.generators_t.p / network.generators.efficiency * network.generators.carrier.map(network.carriers.co2_emissions)
    return emissions.sum().sum() / 1000000

def get_installed_capacity(year):
    network = networks[year]

    capacities = network.generators.groupby(by='carrier')['p_nom_opt'].sum()

    if 'ror' in capacities:
        capacities['hydro'] = capacities.get('hydro', 0) + capacities.pop('ror')

    if 'hydro' in network.storage_units.carrier.unique():
        hydro_capacity = network.storage_units[network.storage_units.carrier == 'hydro']['p_nom_opt'].sum()
        capacities['hydro'] += hydro_capacity

    capacities.pop('load')
    capacities_df = capacities.to_frame().transpose()
    capacities_df.index = [year]

    return capacities_df

def get_installed_capacity_charge(year):
    network = networks[year]
    capacities_discharge = network.links.groupby(by='carrier')['p_nom'].sum()
    capacities_discharge_df = capacities_discharge.to_frame().transpose()
    capacities_discharge_df.index = [year]
    return capacities_discharge_df

def get_installed_capacity_battery(year):
    network = networks[year]
    storage_capacity = network.stores.groupby(by='carrier')['e_nom'].sum()
    storage_capacity_df = storage_capacity.to_frame().transpose()
    storage_capacity_df.index = [year]
    return storage_capacity_df

def get_installed_capacity_lines(year):
    network = networks[year]
    lines_capacity = network.lines.groupby(by='carrier')['s_nom'].sum()
    lines_capacity_df = lines_capacity.to_frame().transpose()
    lines_capacity_df.index = [year]
    return lines_capacity_df

def get_snapshot_generation(year, first_date, second_date, carrier):
    network = networks[year]
    if carrier == 'hydro':
        generation = network.storage_units_t.p_dispatch[first_date:second_date].groupby(network.storage_units.carrier, axis=1).sum()[carrier]
    elif carrier == 'battery':
        generation = network.stores_t.p.loc[first_date:second_date].groupby(network.stores.carrier, axis=1).sum()[carrier]
    else:
        generation = network.generators_t.p.loc[first_date:second_date].groupby(network.generators.carrier, axis=1).sum()[carrier]
    return generation

def get_snapshot_demand(year, first_date, second_date):
    network = networks[year]
    demand = network.loads_t.p_set.loc[first_date:second_date].sum(axis=1)*-1
    return demand

def get_new_installed(years, merge_ror=True):

    data_list = []

    for y in years:
        net = networks[y]

        capacity = net.generators[["p_nom_opt","carrier","p_nom"]]
        hydro = net.storage_units[["p_nom_opt","carrier","p_nom"]]

        caps = pd.concat([capacity, hydro], ignore_index=True)

        caps["p_change"] = caps["p_nom_opt"] - caps["p_nom"]
        caps["year"] = y

        data_list.append(caps[["year","carrier","p_change"]])

    data_agg = pd.concat(data_list)

    grouped = data_agg.groupby(['year','carrier']).sum().unstack()

    grouped.columns = grouped.columns.droplevel(0)
    grouped = grouped.clip(lower=0)
    grouped = grouped.drop('load', axis=1)

    if merge_ror and "ror" in grouped.columns:
        grouped['hydro'] += grouped.pop('ror')

    return grouped

def get_new_installed_battery(years):
    capacity = {'Charger':[], 'Discharger':[],'Battery Storage':[], 'year':[]}

    for y in years:
        net=networks[y]

        capacity["year"].append(y)

        charger_capacity = net.links.groupby('carrier').p_nom.sum().get('battery charger', 0)
        charger_next_capacity = net.links.groupby('carrier').p_nom_opt.sum().get('battery charger', 0)
        capacity['Charger'].append(charger_next_capacity-charger_capacity)

        discharger_capacity = net.links.groupby('carrier').p_nom.sum().get('battery discharger', 0)
        discharger_next_capacity = net.links.groupby('carrier').p_nom_opt.sum().get('battery discharger', 0)
        capacity['Discharger'].append(discharger_next_capacity-discharger_capacity)

        battery_storage_capacity = net.stores.groupby('carrier').e_nom.sum().get('battery', 0)
        battery_storage_next_capacity = net.stores.groupby('carrier').e_nom_opt.sum().get('battery', 0)
        capacity['Battery Storage'].append(battery_storage_next_capacity-battery_storage_capacity)

    capacity_battery_df = pd.DataFrame(capacity)
    capacity_battery_df.set_index("year", inplace=True)

    return capacity_battery_df

def get_new_installed_lines(years):
    data_agg = pd.DataFrame({})

    for y in years:
        net = networks[y]
        lines = pd.DataFrame(net.lines)

        lines["line_id"] = lines.index
        
        lines["p_change"] = lines["s_nom_opt"] - lines["s_nom"]
        lines["year"] = np.ones(len(lines["s_nom_opt"]), dtype=int) * y

        data_agg = pd.concat([data_agg, lines[["year", "line_id", "p_change"]]])

    grouped_cap_change_L = data_agg.groupby(['year', 'line_id']).sum().unstack()

    grouped_cap_change_L.columns = grouped_cap_change_L.columns.droplevel(0)
    grouped_cap_change_L = grouped_cap_change_L.clip(lower=0)
    grouped_cap_change_L = grouped_cap_change_L.sort_index(axis=1)

    return grouped_cap_change_L

def rename_columns(df):
    new_names = ['Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Geothermal', 'Battery']
    old_names = ['biomass', 'oil', 'onwind', 'solar', 'hydro', 'geothermal', 'battery']
    name_map= dict(zip(old_names, new_names))
    df = df.rename(columns=name_map)
    return df

def get_colors(carriers):
    colors = [beige2, beige3, teal3, beige1, teal4, yellow, teal2, brown, brown2]
    names = ['CCGT',    'OCGT',  'Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Battery', 'Geothermal']
    color_dict = dict(zip(names, colors))
    colors_new = [color_dict[carrier] for carrier in carriers]
    return colors_new

def get_marginal_cost(y, carrier):
    network = networks[y]
    carrier_data = network.generators.loc[network.generators['carrier'] == carrier]

    if not carrier_data.empty:
        marginal_cost = carrier_data['marginal_cost'].iloc[0]
        return marginal_cost
    else:
        return 0 

def get_subsidies():
    subsidies = {}

    if cost_loop:
        cost_timeline = [
            (2020, {'CCGT': 47.82, 'OCGT': 65.19, 'oil': 157.35}),
            (2025, {'CCGT': 46.97, 'OCGT': 64.45, 'oil': 157.37}),
            (2030, {'CCGT': 46.13, 'OCGT': 63.73, 'oil': 157.37}),
            (2035, {'CCGT': 45.72, 'OCGT': 63.02, 'oil': 157.36}),
            (2040, {'CCGT': 45.32, 'OCGT': 62.32, 'oil': 157.36}),
        ]
    elif gas_price_1:
        cost_timeline = [
            (2020, {'CCGT': 44.88, 'OCGT': 55.56, 'oil': 128.22}),
            (2030, {'CCGT': 46.74, 'OCGT': 57.94, 'oil': 128.22}),
        ]
    elif gas_price_4:
        cost_timeline = [
            (2020, {'CCGT': 71.15, 'OCGT': 89.34, 'oil': 128.22}),
            (2028, {'CCGT': 76.32, 'OCGT': 95.98, 'oil': 128.22}),
            (2030, {'CCGT': 89.46, 'OCGT': 112.88, 'oil': 128.22}),
            (2035, {'CCGT': 98.01, 'OCGT': 123.87, 'oil': 128.22}),
        ]
    else:
        actual_cost = {'CCGT': 51.01, 'OCGT': 63.44, 'oil': 128.22}
    for y in years:
        if cost_loop or gas_price_1 or gas_price_4:
            for interval_start, costs in reversed(cost_timeline):
                if y >= interval_start:
                    actual_cost = costs
                    break
        for carrier in ['CCGT', 'OCGT', 'oil']:
            marginal_cost= get_marginal_cost(y, carrier)
            subsidies[(y, carrier)] = actual_cost[carrier] - marginal_cost
    return subsidies

def calculate_present_value(future_value, year, base_year, discount_rate):
    return future_value / ((1 + discount_rate) ** (year - base_year))


def capacity_factors(year):
    carrier_mapping = {
        'Biomass': 'biomass',
        'Combined-Cycle Gas': 'CCGT',
        'Oil': 'oil',
        'Onshore Wind': 'onwind',
        'Open-Cycle Gas': 'OCGT',
        'Solar': 'solar',
        'Run of River': 'hydro',
        'Reservoir & Dam': 'hydro_store',
        'Geothermal': 'geothermal'
    }

    network = networks[year]
    network_stats = network.statistics()
    cp = {output: [] for output in carrier_mapping.values()}

    for carrier_stat, carrier in carrier_mapping.items():
        if carrier_stat == 'Reservoir & Dam':
            value = network_stats.loc['StorageUnit']['Capacity Factor'][carrier_stat]
            cp[carrier] = value
        else:
            value = network_stats.loc['Generator']['Capacity Factor'].drop('load')[carrier_stat]
            cp[carrier] = value
            
    # merge hydro and hydro_store into one value using their mean
    if 'hydro_store' in cp:
        cp['hydro'] = (cp['hydro'] + cp['hydro_store']) / 2
        del cp['hydro_store']
        
    return cp

def capcost_lines(network):

    capital_cost_df = pd.DataFrame()
    for line in network.lines.index:
        capital_cost = network.lines.loc[line, 'capital_cost']
        new_row = pd.DataFrame({'capital_cost': capital_cost}, index=[line])
        capital_cost_df = pd.concat([capital_cost_df, new_row])

    return capital_cost_df

def get_operational_costs(years):
    operational_costs_by_year = []

    for year in years:
        power_prod_df = get_power_prod_hydro(year)
        total_operational_cost = 0
        network = networks[year]
        
        for carrier in power_prod_df.columns:
            if carrier == 'hydro': # hydro is a storage unit, not a generator
                production = power_prod_df[carrier].iloc[0]
                operational_cost = network.storage_units.loc[network.storage_units['carrier'] == carrier, 'marginal_cost'].mean() * production
                total_operational_cost += operational_cost                
            
            elif carrier != 'load':  # Skip 'load' as it is not a generation carrier
                production = power_prod_df[carrier].iloc[0]
                operational_cost = network.generators.loc[network.generators['carrier'] == carrier, 'marginal_cost'].mean() * production
                total_operational_cost += operational_cost
        
        operational_costs_by_year.append(total_operational_cost)
    
    return operational_costs_by_year


def get_capital_costs(network):
    capital_cost_df = pd.DataFrame()
    for carrier in network.generators['carrier'].unique():
        if carrier != 'load':
            capital_cost = network.generators.loc[network.generators['carrier'] == carrier, 'capital_cost'].mean()
            new_row = pd.DataFrame({'capital_cost': capital_cost}, index=[carrier])
            capital_cost_df = pd.concat([capital_cost_df, new_row])
    # Add hydro costs
    
    hydro_cost = 270940.71528
    new_row = pd.DataFrame({'capital_cost': hydro_cost}, index=['hydro'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    # Add battery, charger, and discharger costs
    battery_cost = network.stores.loc[network.stores['carrier'] == 'battery', 'capital_cost'].mean()
    charger_cost = network.links.loc[network.links['carrier'] == 'battery charger', 'capital_cost'].mean()
    discharger_cost = network.links.loc[network.links['carrier'] == 'battery discharger', 'capital_cost'].mean()
    new_row = pd.DataFrame({'capital_cost': battery_cost}, index=['battery'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    new_row = pd.DataFrame({'capital_cost': charger_cost}, index=['battery charger'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    new_row = pd.DataFrame({'capital_cost': discharger_cost}, index=['battery discharger'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    
    return capital_cost_df


In [ ]:
def run_scenario(case, case_path, code, discountrate, cost_loop, gas_price_4, gas_price_1):
    """Load networks and save all CSVs for one scenario."""
    print(f"\n{'='*50}\nProcessing: {case}\n{'='*50}")

    # --- load networks (your existing Cell 5) ---
    if case_path == 'base_gas_low' or case_path == 'base_gas_high':
        path_name = path + 'base/' + code
    else:
        path_name = path + case_path + '/' + code
    years = [2024, 2025, 2026, 2027, 2028, 2029,
             2030, 2031, 2032, 2033, 2034, 2035,
             2036, 2037, 2038, 2039, 2040]
    networks = {y: pypsa.Network(path_name + str(y) + '.nc') for y in years}
    first_year, final_year = years[0], years[-1]

    globals()['networks'] = networks


    def _get_subsidies():
        """Local version of get_subsidies() using this scenario's flags."""
        subsidies = {}
        if cost_loop:
            cost_timeline = [
                (2020, {'CCGT': 47.82, 'OCGT': 65.19, 'oil': 157.35}),
                (2025, {'CCGT': 46.97, 'OCGT': 64.45, 'oil': 157.37}),
                (2030, {'CCGT': 46.13, 'OCGT': 63.73, 'oil': 157.37}),
                (2035, {'CCGT': 45.72, 'OCGT': 63.02, 'oil': 157.36}),
                (2040, {'CCGT': 45.32, 'OCGT': 62.32, 'oil': 157.36}),
            ]
        elif gas_price_1:
            cost_timeline = [
                (2020, {'CCGT': 44.88, 'OCGT': 55.56, 'oil': 128.22}),
                (2030, {'CCGT': 46.74, 'OCGT': 57.94, 'oil': 128.22}),
            ]
        elif gas_price_4:
            cost_timeline = [
                (2020, {'CCGT': 71.15, 'OCGT':  89.34, 'oil': 128.22}),
                (2028, {'CCGT': 76.32, 'OCGT':  95.98, 'oil': 128.22}),
                (2030, {'CCGT': 89.46, 'OCGT': 112.88, 'oil': 128.22}),
                (2035, {'CCGT': 98.01, 'OCGT': 123.87, 'oil': 128.22}),
            ]
        else:
            actual_cost = {'CCGT': 51.01, 'OCGT': 63.44, 'oil': 128.22}

        for y in years:
            if cost_loop or gas_price_1 or gas_price_4:
                for interval_start, costs in reversed(cost_timeline):
                    if y >= interval_start:
                        actual_cost = costs
                        break
            for carrier in ['CCGT', 'OCGT', 'oil']:
                marginal_cost = get_marginal_cost(y, carrier)  # still uses global helper
                subsidies[(y, carrier)] = actual_cost[carrier] - marginal_cost
        return subsidies

    # ── COST CALCULATIONS ──────────────────────────────────────────────────
    new_installed_cap   = get_new_installed(years, merge_ror=False)
    new_installed_bat   = get_new_installed_battery(years)
    new_installed_lines = get_new_installed_lines(years)

    capital_costs_final      = []
    generator_costs_final    = []
    line_costs_final         = []
    operational_costs_by_year = []
    yearly_installed_lines   = {}

    for y in years:
        network = networks[y]
        capital_cost_df    = get_capital_costs(network)
        capital_cost_lines = capcost_lines(network)

        total_gen_capex = total_bat_capex = total_line_capex = total_capex = 0

        for carrier in new_installed_cap.columns:
            capex_value = new_installed_cap.loc[:y, carrier].sum() * capital_cost_df.loc[carrier, 'capital_cost']
            total_capex    += capex_value
            total_gen_capex += capex_value

        bat_capex  = new_installed_bat.loc[:y, 'Battery Storage'].sum() * capital_cost_df.loc['battery',            'capital_cost']
        bat_capex += new_installed_bat.loc[:y, 'Charger'].sum()         * capital_cost_df.loc['battery charger',    'capital_cost']
        bat_capex += new_installed_bat.loc[:y, 'Discharger'].sum()      * capital_cost_df.loc['battery discharger', 'capital_cost']
        total_bat_capex += bat_capex
        total_capex     += bat_capex

        for line in new_installed_lines.columns:
            line_capex = new_installed_lines.loc[:y, str(line)].sum() * capital_cost_lines.loc[line, 'capital_cost']
            total_capex     += line_capex
            total_line_capex += line_capex

        yearly_installed_lines[y] = new_installed_lines.loc[y, :].sum()

        capital_costs_final.append(total_capex / 1e6)
        generator_costs_final.append((total_gen_capex + total_bat_capex) / 1e6)
        line_costs_final.append(total_line_capex / 1e6)

    operational_costs_by_year = get_operational_costs(years)

    pv_gen_capex          = [calculate_present_value(i, y, years[0], discountrate) for i, y in zip(generator_costs_final,    years)]
    pv_line_capex         = [calculate_present_value(i, y, years[0], discountrate) for i, y in zip(line_costs_final,         years)]
    pv_capital_costs      = [calculate_present_value(i, y, years[0], discountrate) for i, y in zip(capital_costs_final,      years)]
    pv_operational_costs  = [calculate_present_value(i, y, years[0], discountrate) for i, y in zip(operational_costs_by_year, years)]
    total_pv              = [c + o for c, o in zip(pv_capital_costs, pv_operational_costs)]

    print(f"  Cap cost: {sum(pv_capital_costs):.1f}  |  Opex: {sum(pv_operational_costs):.1f}  |  NPV: {sum(total_pv):.1f}")

    # ── SUBSIDIES ──────────────────────────────────────────────────────────
    subsidies = _get_subsidies()   # <-- uses the local version, not the global
    cost_of_subsidies = {}

    for year in years:
        power_prod = get_power_prod(year)
        for (subsidy_year, carrier), subsidy_rate in subsidies.items():
            if subsidy_year != year:
                continue
            carrier_name = 'Oil' if carrier == 'oil' else carrier
            if carrier_name not in power_prod.columns or power_prod.empty:
                continue
            cost_of_subsidies.setdefault(year, {}).setdefault(carrier_name, 0)
            cost_of_subsidies[year][carrier_name] += subsidy_rate * power_prod[carrier_name].iloc[0]

    cost_of_subsidies_df = rename_columns(pd.DataFrame.from_dict(cost_of_subsidies, orient='index'))
    subsidies_list = list(cost_of_subsidies_df.sum(axis=1))
    pv_subsidies   = [calculate_present_value(i, y, years[0], discountrate) for i, y in zip(subsidies_list, years)]
    npv_subsidies  = sum(pv_subsidies)

    # ── SAVE TO CSV ────────────────────────────────────────────────────────
    if save_to_csv:

        # helpers for the read-append-write pattern
        def _append_column(filename, column_name, values):
            try:
                df = pd.read_csv(filename)
            except FileNotFoundError:
                df = pd.DataFrame()
            df[column_name] = values
            df.to_csv(filename, index=False)

        _append_column('../result_data/capcost.csv',      case_path, pv_capital_costs)
        _append_column('../result_data/opex.csv',         case_path, pv_operational_costs)
        _append_column('../result_data/gencost.csv',      case_path, pv_gen_capex)
        _append_column('../result_data/linecost.csv',     case_path, pv_line_capex)
        _append_column('../result_data/total_costs.csv',  case_path, total_pv)
        _append_column('../result_data/subsidies.csv',    case_path, [npv_subsidies])
        _append_column('../result_data/emissions.csv',    case_path,
                       [get_emissions(y) for y in years])

        pd.DataFrame.from_dict(yearly_installed_lines, orient='index',
                               columns=['Total_capacity']
                               ).to_csv('../result_data/' + case_path + '_new_line_capacity.csv')

        get_installed_capacity(first_year).pipe(rename_columns).to_csv(
            '../result_data/' + case_path + '_24_final_capacity.csv')
        get_installed_capacity(final_year).pipe(rename_columns).to_csv(
            '../result_data/' + case_path + '_final_capacity.csv')

        get_new_installed(years, merge_ror=True).pipe(rename_columns).to_csv(
            '../result_data/' + case_path + '_new_capacity.csv')

        get_new_installed_battery(years).to_csv(
            '../result_data/' + case_path + '_new_battery_capacity.csv')

        pd.concat([get_power_prod(y) for y in years]
                  ).set_index(pd.Index(years)).astype('float32').pipe(rename_columns).to_csv(
            '../result_data/' + case_path + '_production.csv')

        cp = pd.DataFrame({y: capacity_factors(y) for y in years}).T
        cp = rename_columns(cp)

        cap = pd.concat([get_installed_capacity(y) for y in years])
        cap = rename_columns(cap)

        cp.index = cp.index.astype(int)
        cap.index = cap.index.astype(int)
        cp = cp.mask(cap.reindex(index=cp.index, columns=cp.columns).fillna(0) < 1e-3, 0)

        cp.to_csv('../result_data/' + case_path + '_capacity_factors.csv')

        get_power_mix(final_year).pipe(rename_columns).to_csv(
            '../result_data/' + case_path + '_power_mix.csv')

        # snapshots
        first_date, second_date = "2013-12-28", "2013-12-31"
        Battery  = get_snapshot_generation(final_year, first_date, second_date, 'battery')
        snapshot = {
            'CCGT':       get_snapshot_generation(final_year, first_date, second_date, 'CCGT'),
            'OCGT':       get_snapshot_generation(final_year, first_date, second_date, 'OCGT'),
            'Oil':        get_snapshot_generation(final_year, first_date, second_date, 'oil'),
            'Geothermal': get_snapshot_generation(final_year, first_date, second_date, 'geothermal'),
            'Hydro':      get_snapshot_generation(final_year, first_date, second_date, 'ror')
                        + get_snapshot_generation(final_year, first_date, second_date, 'hydro'),
            'Wind':       get_snapshot_generation(final_year, first_date, second_date, 'onwind'),
            'Solar':      get_snapshot_generation(final_year, first_date, second_date, 'solar'),
            'Biomass':    get_snapshot_generation(final_year, first_date, second_date, 'biomass'),
            'Battery':    [i if i > 0 else 0 for i in Battery],
            'Nbattery':   [i if i < 0 else 0 for i in Battery],
            'Demand':     get_snapshot_demand(final_year, first_date, second_date),
            'Lost Load':         networks[final_year].generators_t.p.loc[first_date:second_date].groupby(networks[final_year].generators.carrier, axis=1).sum()['load']/1e3
        }
        pd.DataFrame(snapshot).to_csv('../result_data/' + case_path + '_snapshots.csv')

    plt.close('all')   # suppress plots during batch run
    print(f"  Saved: {case}")


# ── Run all scenarios ──────────────────────────────────────────────────────
if save_to_csv:
    import os
    os.makedirs('../result_data', exist_ok=True)

for scenario_args in SCENARIOS:
    run_scenario(*scenario_args)

print("\nAll scenarios complete.")